#### Statistical Data Analysis

Dataset: 

- _music_clean.csv_  

Author: Luis Sergio Pastrana Lemus  
Date: 2025-04-23

# Statistical Data Analysis – Music Activity Dataset

## __1. Libraries__

In [8]:
from IPython.display import display, HTML
import os
import pandas as pd
from pathlib import Path
import scipy.stats as st
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import sys


# Define project root dynamically, gets the current directory from whick the notebook belongs and moves one level upper
project_root = Path.cwd().parent

# Add src to sys.path if it is not already
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Import function directly (more controlled than import *)
from src import *

## __2. Path to Data file__

In [2]:
# Build route to data file and upload
data_file_path = project_root / "data" / "processed"
df_music = load_dataset_from_csv(data_file_path, "music_clean.csv", sep=',', header='infer')

## __3. Statistical Data Analysis__

### 3.1  inferential statistical tests


#### 3.1.1  Hypothesis testing: User activity varies by city

In [3]:
# Hypothesis: User activity varies by city.

# 1. Propose Hypotheses H0, H1
# H0: User activity does not vary by city, user activity is the same (==)
# H1: User activity varies by city, user activity is different (!=)

# Prepare data by city for t-test
df_music['track_count'] = 1
city_grouped = df_music.groupby(['city', 'userid'])['track_count'].sum().reset_index()

springfield_music_activity = city_grouped.loc[(city_grouped['city'] == 'springfield'), 'track_count']
shelbyville_music_activity = city_grouped.loc[(city_grouped['city'] == 'shelbyville'), 'track_count']

# 2. Specify Significance or Confidence
# alpha = 5%
# confidence = 95%

alpha = 0.05

In [4]:
# Levene's test, to ensure that the variances of different samples are equal. 
# Preventing Tests Like ANOVA and t-Tests from Being Incorrect

levene_stat, levene_p = st.levene(springfield_music_activity, shelbyville_music_activity)
display(HTML(f"<b>Levene's Test</b> – Statistic: {levene_stat:.4f}, P-value: {levene_p:.4f}"))

# Determining Equality of Variances
if levene_p < 0.05:
    equal_var = False
    display(HTML("<i>Null Hypothesis H₀ is rejected: the variances are different → use equal_var=False</i>"))
else:
    equal_var = True
    display(HTML("<i>Null Hypothesis H₀ is not rejected: the variances are equal → use equal_var=True</i>"))

In [ ]:
# 3. Calculate critical and test values, define acceptance and rejection zones

t_stat_city, p_val_city = st.ttest_ind(springfield_music_activity, shelbyville_music_activity, equal_var=False)

display(HTML(f"T-statistic: <b>{t_stat_city:.4f}</b>"))
display(HTML(f"P-value: <b>{p_val_city:.4f}</b>"))

# 4. Decision and Conclusion

if p_val_city < alpha:
    display(HTML("The <i>'H₀ null hypothesis'</i> is <b>rejected</b>, <b>not rejecting</b> <i>'H₁ alternative hypothesis'</i>, because there is sufficient statistical evidence to affirm that <b> user activity differs by city.</b>"))
else:
    display(HTML("The <i>'H₀ null hypothesis'</i> is <b>not rejected</b>, <b>rejecting <i>'H₁ alternative hypothesis'</i>, because there is sufficient statistical evidence to affirm that <b> user activity does not differ by city</b>."))

#### 3.1.2 Hypothesis testing: User activity varies by day of week and city

In [7]:
# Hypothesis: User activity varies by day of week and city.

# 1. Propose Hypotheses H0, H1
# H0: User activity does not vary by day of week and city, user activity is the same (==)
# H1: User activity varies by day of week and city, user activity is different (!=)

# Prepare data by city for ANOVA
day_city_grouped = df_music.groupby(['city', 'day', 'userid'])['track_count'].sum().reset_index()

# 2. Specify Significance or Confidence
# alpha = 5%
# confidence = 95%

alpha = 0.05

In [ ]:
# 3. Calculate critical and test values, define acceptance and rejection zones

# Run two-way ANOVA
model = ols('track_count ~ C(city) + C(day) + C(city):C(day)', data=day_city_grouped).fit()
anova_table = sm.stats.anova_lm(model, typ=2) # Parametric ANOVA assuming Residuals Normality, Homogeneity of variances, Independence of observations.

# 4. Decision and Conclusion

# City effect
if anova_table.loc['C(city)', 'PR(>F)'] < alpha:
    display(HTML("The <i>'H₀ null hypothesis'</i> is <b>rejected</b>, <b>not rejecting</b> <i>'H₁ alternative hypothesis'</i>, because there is sufficient statistical evidence to affirm that <b>user activity differs by city.</b>"))
else:
    display(HTML("The <i>'H₀ null hypothesis'</i> is <b>not rejected</b>, <b>rejecting</b> <i>'H₁ alternative hypothesis'</i>, because there is sufficient statistical evidence to affirm that <b>user activity does not differ by city</b>."))

# Day effect
if anova_table.loc['C(day)', 'PR(>F)'] < alpha:
    display(HTML("The <i>'H₀ null hypothesis'</i> is <b>rejected</b>, <b>not rejecting</b> <i>'H₁ alternative hypothesis'</i>, because there is sufficient statistical evidence to affirm that <b>user activity differs by day of week.</b>"))
else:
    display(HTML("The <i>'H₀ null hypothesis'</i> is <b>not rejected</b>, <b>rejecting</b> <i>'H₁ alternative hypothesis'</i>, because there is sufficient statistical evidence to affirm that <b>user activity does not differ by day of week</b>."))

# Interaction effect
if anova_table.loc['C(city):C(day)', 'PR(>F)'] < alpha:
    display(HTML("The <i>'H₀ null hypothesis'</i> is <b>rejected</b>, <b>not rejecting</b> <i>'H₁ alternative hypothesis'</i>, because there is sufficient statistical evidence to affirm that <b>there is an interaction effect between city and day on user music activity</b>"))
else:
    display(HTML("The <i>'H₀ null hypothesis'</i> is <b>not rejected</b>, <b>rejecting</b> <i>'H₁ alternative hypothesis'</i>, because there is sufficient statistical evidence to affirm that <b>there is not an interaction effect between city and day on user music activity</b>."))
    

## 4. Conclusion of Statistical Data Analysis – Music Activity

This analysis aimed to assess user music activity patterns across different cities and days of the week.

Descriptive Statistics highlighted clear differences in user behavior, with Springfield consistently showing higher user engagement metrics compared to Shelbyville.

A T-test confirmed that the difference in total tracks between Springfield and Shelbyville is statistically significant, indicating user activity is not equal across cities.

The ANOVA results may not align with the exploratory analysis because the data distribution violates key assumptions of the test. The variable track_count shows extremely low dispersion (IQR = 0) and a high presence of outliers, indicating a non-normal and heavily skewed distribution. Under these conditions, ANOVA loses validity, and a non-parametric alternative such as the Kruskal–Wallis test is more appropriate.
